# VAT (Virus, AI, Test) — Entrenamiento del clasificador de texto

Este notebook entrena **tu propio clasificador** de 4 clases: `human`, `chatgpt`,
`claude`, `gemini`. Corré cada celda en orden con Shift+Enter (o "Entorno de
ejecución > Ejecutar todas").

**Qué necesitás tener listo antes de arrancar:** solo 3 archivos —
`chatgpt.txt`, `claude.txt`, `gemini.txt` (mismas 50 consignas, separadas por
líneas `---`). El texto humano lo trae el notebook solo, gratis, desde un
dataset público de noticias en español — no hace falta que juntes nada para esa clase.

**Antes de empezar:** anda a `Entorno de ejecución > Cambiar tipo de entorno de
ejecución` y elegí **GPU** (T4 gratis). Sin GPU, el entrenamiento igual funciona
pero tarda mucho más — en CPU podés estar esperando 10x más tiempo.

In [ ]:
# ============================================================
# CELDA 1 — Instalar dependencias
# ============================================================
# Colab ya trae Python y algunas librerías, pero NO trae "transformers"
# (la librería de Hugging Face para cargar y entrenar modelos) ni "datasets"
# (para manejar los datos de entrenamiento de forma eficiente).
#
# Qué pasaría si no corrés esta celda: las siguientes celdas van a tirar
# "ModuleNotFoundError: No module named transformers" y el notebook entero
# se corta ahí. Es la base de todo lo demás.
!pip install -q transformers datasets accelerate scikit-learn

In [ ]:
# ============================================================
# CELDA 2 — Subir los 3 archivos de texto (IA) — el humano sale gratis solo
# ============================================================
# files.upload() abre una ventana para que subas archivos desde tu compu.
# Subí solo 3 ahora: chatgpt.txt, claude.txt, gemini.txt
#
# El texto HUMANO no lo subís vos: lo trae automáticamente la Celda 3-B
# desde un dataset público y gratuito de noticias en español (~2000
# artículos, de cientos de periodistas distintos). Esto es mejor que
# pedirle textos a 3-4 conocidos: más volumen y más diversidad de autores,
# sin que te cueste plata ni tiempo de conseguirlo.
#
# Por qué está separado de la lectura: si combináramos "subir" y "leer" en
# la misma celda, cada vez que quieras re-ejecutar algo más abajo (por un
# error de tipeo, por ejemplo) tendrías que volver a subir los archivos.
#
# Qué pasaría si no corrés esta celda: la Celda 3 (que lee los archivos)
# no va a encontrar los archivos en el disco de Colab y tira
# "FileNotFoundError".
from google.colab import files

print("Subí los 3 archivos: chatgpt.txt, claude.txt, gemini.txt")
subidos = files.upload()
print("Archivos recibidos:", list(subidos.keys()))

In [ ]:
# ============================================================
# CELDA 3 — Función para leer cada archivo y separarlo en ejemplos
# ============================================================

def leer_textos_de_archivo(ruta_archivo):
    """
    Lee un archivo .txt y lo separa en una lista de textos individuales,
    usando líneas que contienen solo "---" (tres o más guiones) como
    separador entre un ejemplo y el siguiente.

    CÓMO SE LEE: abre el archivo en modo texto con codificación UTF-8
    (necesario para que tildes y "ñ" no se rompan), lee todo el contenido
    de una vez, y usa una expresión regular para cortar el texto en cada
    línea que sea exclusivamente guiones.

    POR QUÉ SE USA ESTE FORMATO: separar por "---" es simple de escribir
    a mano en cualquier editor de texto, no requiere estructura JSON ni
    CSV (donde una coma o comilla mal puesta rompe todo el archivo), y es
    fácil de revisar visualmente si algo salió mal.

    QUÉ PASARÍA SI ESTA FUNCIÓN NO EXISTIERA: tendríamos que separar los
    textos a mano, copiando y pegando cada uno en variables individuales.
    Con 50 textos por archivo, eso es lento y muy propenso a errores
    (por ejemplo, olvidarse de un texto o pegarlo en la clase equivocada).

    QUÉ PASARÍA SI EL ARCHIVO NO TUVIERA NINGÚN "---": la función
    devolvería una lista con un solo elemento (todo el archivo junto como
    si fuera un solo texto gigante) — por eso más abajo avisamos si algún
    archivo trae muy pocos textos, para detectar este error temprano.
    """
    import re
    with open(ruta_archivo, "r", encoding="utf-8") as f:
        contenido = f.read()

    # re.split corta el texto en cada línea que sea SOLO guiones (3 o más).
    # El flag re.MULTILINE hace que ^ y $ reconozcan cada línea del archivo,
    # no solo el principio/final del archivo completo.
    partes = re.split(r"^-{3,}\s*$", contenido, flags=re.MULTILINE)

    # strip() saca espacios/saltos de línea sobrantes al principio y final
    # de cada texto. El filtro descarta partes vacías (por ejemplo si el
    # archivo termina con un "---" suelto sin texto después).
    textos = [p.strip() for p in partes if p.strip() != ""]
    return textos


# Cargamos los 3 archivos de IA. Los nombres deben coincidir EXACTO con lo
# que subiste en la Celda 2 (mismo nombre, mismas mayúsculas/minúsculas).
textos_chatgpt = leer_textos_de_archivo("chatgpt.txt")
textos_claude  = leer_textos_de_archivo("claude.txt")
textos_gemini  = leer_textos_de_archivo("gemini.txt")

print(f"ChatGPT: {len(textos_chatgpt)} textos")
print(f"Claude:  {len(textos_claude)} textos")
print(f"Gemini:  {len(textos_gemini)} textos")

for nombre, lista in [("chatgpt", textos_chatgpt), ("claude", textos_claude),
                       ("gemini", textos_gemini)]:
    if len(lista) < 15:
        print(f"⚠️  {nombre} tiene pocos textos ({len(lista)}). Conviene juntar más antes de entrenar.")

In [ ]:
# ============================================================
# CELDA 3-B — Traer texto HUMANO real (dataset público + tu human.txt opcional)
# ============================================================
# Usamos "mteb/SpanishNewsClassification": 2048 artículos de noticias reales
# en español, licencia MIT, columnas "text" y "label". (Este nombre está
# verificado. El que usábamos antes, "MarcOrfilaCarreras/spanish-news",
# ya no existe en el Hub — por eso te tiró DatasetNotFoundError.)
#
# QUÉ PASARÍA SI NO CORRIÉRAMOS ESTA CELDA: no habría ejemplos de la clase
# "human" y el modelo no podría aprender a distinguir humano de IA.

from datasets import load_dataset
import random, re, os

DATASET_HUMANO = "mteb/SpanishNewsClassification"
CANTIDAD_HUMANO = 150   # cuántos artículos tomar (parecido a las otras clases)

dataset_noticias = load_dataset(DATASET_HUMANO, split="train")

def recortar_a_largo(texto, largo_objetivo):
    """
    Recorta un artículo a un largo parecido al de los textos de IA, cortando
    en un final de oración (. ? !) para que no quede una frase cortada a la mitad.

    POR QUÉ HACE FALTA: los artículos de noticias miden desde 500 hasta más de
    200.000 caracteres, y tus respuestas de IA miden entre 170 y ~1.400. Si
    dejáramos los artículos completos, el modelo aprendería el atajo
    "texto largo = humano" en vez del estilo real de escritura. Es una trampa
    (fuga de datos) que hace que el accuracy se vea lindo en el entrenamiento
    y falle cuando alguien pegue un texto corto de verdad.

    QUÉ PASARÍA SI NO EXISTIERA: el modelo sacaría buenas métricas por la razón
    equivocada, y en producción confundiría cualquier texto corto con IA.
    """
    texto = re.sub(r"\s+", " ", texto).strip()
    if len(texto) <= largo_objetivo:
        return texto
    corte = texto[:largo_objetivo]
    ultimo_punto = max(corte.rfind(". "), corte.rfind("? "), corte.rfind("! "))
    if ultimo_punto > largo_objetivo * 0.5:
        return corte[:ultimo_punto + 1]
    return corte

# Largos de los textos de IA que ya cargamos en la Celda 3. Cada artículo
# humano se recorta a un largo tomado al azar de esta lista, así las dos
# clases quedan con la MISMA distribución de largos.
largos_ia = [len(t) for t in (textos_chatgpt + textos_claude + textos_gemini)]

random.seed(42)
indices = random.sample(range(len(dataset_noticias)), min(len(dataset_noticias), CANTIDAD_HUMANO * 2))

textos_human = []
for i in indices:
    texto = dataset_noticias[i]["text"].strip()
    # Descartamos artículos con basura de página web (pies de foto repetidos)
    # y quitamos el "Enlace copiado" que algunos traen al principio.
    if texto.count("foto:") > 1:
        continue
    texto = texto.replace("Enlace copiado", "", 1).strip()
    texto = recortar_a_largo(texto, random.choice(largos_ia))
    if len(texto) > 150:
        textos_human.append(texto)
    if len(textos_human) >= CANTIDAD_HUMANO:
        break

print(f"Human (dataset público, ya recortado): {len(textos_human)} textos")

# --- OPCIONAL: sumar textos humanos propios ---------------------------------
# Si subiste un human.txt en la Celda 2 (textos que escribieron personas
# reales: tuyos, de compañeros, mensajes largos), se suman a los de arriba.
# Ayuda mucho: las noticias son un estilo muy formal, y con textos más
# cotidianos el modelo distingue mejor "humano" de "IA" en general.
if os.path.exists("human.txt"):
    propios = leer_textos_de_archivo("human.txt")
    textos_human += propios
    print(f"+ {len(propios)} textos humanos propios (human.txt). Total human: {len(textos_human)}")

In [ ]:
# ============================================================
# CELDA 4 — Armar el dataset combinado (texto + etiqueta numérica)
# ============================================================
# Un modelo de clasificación no entiende las palabras "chatgpt" o "claude"
# como etiquetas: necesita números. Por eso mapeamos cada clase a un
# número entero (0, 1, 2, 3) y guardamos ese mapeo para poder traducir
# los resultados de vuelta a texto legible más adelante.
#
# QUÉ PASARÍA SI NO EXISTIERA ESTE MAPEO: el modelo no podría entrenarse
# en absoluto, porque su capa de salida es matemáticamente un vector de
# números (probabilidades por clase), no texto.

etiquetas = {0: "human", 1: "chatgpt", 2: "claude", 3: "gemini"}
etiquetas_inversas = {v: k for k, v in etiquetas.items()}

# Armamos dos listas paralelas: una con todos los textos, otra con la
# etiqueta numérica correspondiente en la misma posición.
todos_los_textos = []
todas_las_etiquetas = []

for texto in textos_human:
    todos_los_textos.append(texto)
    todas_las_etiquetas.append(etiquetas_inversas["human"])

for texto in textos_chatgpt:
    todos_los_textos.append(texto)
    todas_las_etiquetas.append(etiquetas_inversas["chatgpt"])

for texto in textos_claude:
    todos_los_textos.append(texto)
    todas_las_etiquetas.append(etiquetas_inversas["claude"])

for texto in textos_gemini:
    todos_los_textos.append(texto)
    todas_las_etiquetas.append(etiquetas_inversas["gemini"])

print(f"Total de ejemplos: {len(todos_los_textos)}")

# train_test_split separa los datos en dos grupos: "entrenamiento" (con el
# que el modelo aprende) y "prueba" (que el modelo NUNCA ve durante el
# entrenamiento, y que usamos después para medir qué tan bien aprendió
# de verdad, no solo si memorizó los ejemplos).
#
# stratify=todas_las_etiquetas asegura que la proporción de cada clase sea
# la misma en ambos grupos. Sin esto, por azar podríamos terminar con un
# grupo de prueba que tiene 0 ejemplos de "gemini", y no podríamos saber
# si el modelo aprendió a reconocerlo o no.
from sklearn.model_selection import train_test_split

textos_train, textos_test, etiquetas_train, etiquetas_test = train_test_split(
    todos_los_textos,
    todas_las_etiquetas,
    test_size=0.2,       # 20% para probar, 80% para entrenar
    random_state=42,     # semilla fija: mismo resultado cada vez que se corre
    stratify=todas_las_etiquetas,
)

print(f"Entrenamiento: {len(textos_train)} ejemplos")
print(f"Prueba:        {len(textos_test)} ejemplos")

In [ ]:
# ============================================================
# CELDA 5 — Tokenización con BETO (modelo base en español)
# ============================================================
# Un modelo de lenguaje no lee "palabras": lee números que representan
# fragmentos de palabras (tokens). El tokenizador es el traductor entre
# texto humano y esos números, y tiene que ser EXACTAMENTE el mismo que
# se usó para pre-entrenar el modelo base — si usáramos un tokenizador
# distinto, los números no significarían lo mismo para el modelo y
# entrenaría sobre datos esencialmente aleatorios.
#
# Usamos "dccuchile/bert-base-spanish-wwm-cased" (conocido como BETO):
# es un modelo BERT pre-entrenado específicamente con texto en español,
# a diferencia del roberta en inglés que usábamos antes para detectar
# ChatGPT. Esto es lo que resuelve el pedido de "que funcione en español".
#
# QUÉ PASARÍA SI USÁRAMOS UN MODELO EN INGLÉS: el modelo entendería mal
# la estructura del español (tildes, "ñ", conjugaciones), partiendo
# palabras de forma rara y perdiendo señal útil para distinguir estilos.

from transformers import AutoTokenizer

MODELO_BASE = "dccuchile/bert-base-spanish-wwm-cased"
tokenizer = AutoTokenizer.from_pretrained(MODELO_BASE)

def tokenizar(lista_de_textos):
    """
    Convierte una lista de textos en los tensores numéricos que el modelo
    necesita para funcionar.

    padding=True: hace que todos los textos del lote tengan la misma
    longitud (rellenando con un token especial "vacío" los más cortos),
    porque el modelo procesa los ejemplos en bloques de tamaño fijo.

    truncation=True + max_length=512: corta los textos que sean más
    largos que 512 tokens. BETO no puede procesar textos más largos que
    eso de una sola vez.

    QUÉ PASARÍA SIN TRUNCATION: un texto muy largo (por ejemplo, una
    respuesta larga de Gemini) tiraría un error en pleno entrenamiento
    en vez de simplemente recortarse, y el notebook se cortaría ahí.
    """
    return tokenizer(lista_de_textos, padding=True, truncation=True, max_length=512)

tokens_train = tokenizar(textos_train)
tokens_test  = tokenizar(textos_test)

print("Ejemplo de texto tokenizado (primeros 20 números):")
print(tokens_train["input_ids"][0][:20])

In [ ]:
# ============================================================
# CELDA 6 — Armar el Dataset en el formato que espera el Trainer
# ============================================================
# La librería "datasets" de Hugging Face tiene su propia clase Dataset,
# optimizada para entrenar de forma eficiente (carga en lotes, no todo
# en memoria de una vez). El Trainer (Celda 8) espera recibir objetos de
# este tipo, no listas de Python comunes.
#
# QUÉ PASARÍA SI LE PASÁRAMOS LISTAS COMUNES AL TRAINER: tiraría un error
# de tipo, porque el Trainer necesita poder acceder a los datos por lotes
# (batches) de forma indexada y en un formato que PyTorch entienda.

import torch

class DatasetVAT(torch.utils.data.Dataset):
    """
    Envoltorio simple que junta los tokens de un texto con su etiqueta,
    en el formato exacto que el Trainer de Hugging Face necesita: un
    diccionario por ejemplo, con las claves de los tokens más "labels".

    __len__ le dice al Trainer cuántos ejemplos hay en total (para saber
    cuándo terminó una "época" de entrenamiento).

    __getitem__ le da al Trainer UN ejemplo puntual cuando lo pide, por
    índice. Sin este método, el Trainer no podría iterar sobre el dataset.

    QUÉ PASARÍA SI FALTARA ESTA CLASE: no habría forma de conectar los
    tokens (Celda 5) con las etiquetas (Celda 4) en un solo objeto que
    el Trainer pueda recorrer — tendríamos los datos, pero sueltos.
    """
    def __init__(self, tokens, etiquetas):
        self.tokens = tokens
        self.etiquetas = etiquetas

    def __len__(self):
        return len(self.etiquetas)

    def __getitem__(self, idx):
        item = {clave: torch.tensor(valor[idx]) for clave, valor in self.tokens.items()}
        item["labels"] = torch.tensor(self.etiquetas[idx])
        return item

dataset_train = DatasetVAT(tokens_train, etiquetas_train)
dataset_test  = DatasetVAT(tokens_test, etiquetas_test)

In [ ]:
# ============================================================
# CELDA 7 — Cargar el modelo base y prepararlo para 4 clases
# ============================================================
# AutoModelForSequenceClassification carga BETO pre-entrenado y le agrega
# una "cabeza" de clasificación nueva (una capa final que no existía en
# el modelo original) con exactamente 4 salidas, una por clase.
#
# num_labels=4 es lo que le dice al modelo cuántas categorías tiene que
# aprender a distinguir. Si pusiéramos 2, el modelo esperaría un problema
# binario y no podría entrenarse con nuestras 4 etiquetas.
#
# QUÉ PASARÍA SI NO ESPECIFICÁRAMOS num_labels: por defecto asume 2
# clases, y al intentar entrenar con etiquetas 0-3 tiraría un error de
# "índice fuera de rango" en la primera pasada de entrenamiento.

from transformers import AutoModelForSequenceClassification

modelo = AutoModelForSequenceClassification.from_pretrained(
    MODELO_BASE,
    num_labels=4,
    id2label=etiquetas,              # para que el modelo "sepa" los nombres
    label2id=etiquetas_inversas,     # y los guarde junto con sus pesos
)

print("Modelo cargado. Cabeza de clasificación lista para 4 clases:", etiquetas)

In [ ]:
# ============================================================
# CELDA 8 — Función de métricas (para saber si el modelo aprende bien)
# ============================================================
# Sin esta función, el Trainer entrenaría igual, pero nunca sabríamos
# QUÉ TAN BIEN está funcionando el modelo en datos que nunca vio (el
# grupo de prueba) — solo veríamos el error de entrenamiento, que puede
# bajar aunque el modelo esté memorizando en vez de aprender de verdad.

import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def calcular_metricas(prediccion_eval):
    """
    El Trainer llama a esta función automáticamente después de cada
    evaluación, pasándole las predicciones crudas del modelo y las
    etiquetas verdaderas del grupo de prueba.

    logits.argmax(-1): el modelo devuelve 4 números por texto (uno por
    clase, no una probabilidad directamente). argmax elige el índice del
    número más alto, es decir, la clase que el modelo considera más
    probable.

    accuracy: porcentaje de textos donde el modelo acertó la clase exacta.

    f1 (promedio "macro"): mide precisión y cobertura combinadas, tratando
    a las 4 clases por igual sin importar si alguna tiene menos ejemplos
    que las demás. Es más informativo que el accuracy solo cuando las
    clases no están perfectamente balanceadas (por ejemplo, si terminamos
    con 45 textos humanos pero 50 de cada IA).

    QUÉ PASARÍA SI NO CALCULÁRAMOS f1 Y SOLO MIRÁRAMOS accuracy: podríamos
    creer que el modelo anda bien cuando en realidad está fallando mucho
    en una sola clase minoritaria, porque el accuracy general la "esconde".
    """
    logits, etiquetas_verdaderas = prediccion_eval
    predicciones = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(etiquetas_verdaderas, predicciones),
        "f1_macro": f1_score(etiquetas_verdaderas, predicciones, average="macro"),
    }

In [ ]:
# ============================================================
# CELDA 9 — Configurar el entrenamiento
# ============================================================
# TrainingArguments centraliza TODOS los parámetros que controlan cómo
# se entrena el modelo. Sin este objeto, tendríamos que pasarle cada
# parámetro suelto al Trainer, y sería fácil olvidar alguno importante.

from transformers import TrainingArguments, Trainer

argumentos_entrenamiento = TrainingArguments(
    output_dir="./resultados_vat",

    # num_train_epochs: cuántas veces el modelo recorre TODO el dataset
    # de entrenamiento. Con datasets chicos (como el nuestro, ~150-200
    # ejemplos) conviene un número bajo: muchas épocas con pocos datos
    # hacen que el modelo memorice los ejemplos en vez de generalizar
    # (esto se llama "overfitting").
    num_train_epochs=6,

    # Tamaño de lote: cuántos ejemplos procesa el modelo antes de
    # ajustar sus pesos una vez. Con GPU gratuita de Colab, 8 suele
    # andar bien sin quedarse sin memoria.
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    # learning_rate: qué tan grandes son los ajustes que hace el modelo
    # en cada paso. Un valor típico y seguro para fine-tuning de BERT.
    learning_rate=2e-5,

    # Evalúa contra el grupo de prueba al final de cada época, así
    # podemos ver si el modelo mejora o empieza a empeorar (señal de
    # overfitting) antes de que termine todo el entrenamiento.
    eval_strategy="epoch",
    save_strategy="epoch",

    # Se queda con el mejor modelo visto durante el entrenamiento según
    # f1_macro, no necesariamente el de la última época.
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",

    logging_steps=10,
    report_to="none",  # evita que pida configurar Weights & Biases
)

entrenador = Trainer(
    model=modelo,
    args=argumentos_entrenamiento,
    train_dataset=dataset_train,
    eval_dataset=dataset_test,
    compute_metrics=calcular_metricas,
)

print("Entrenador configurado. Listo para entrenar en la próxima celda.")

In [ ]:
# ============================================================
# CELDA 10 — Entrenar (acá tarda unos minutos)
# ============================================================
# Esta línea dispara el entrenamiento real: el modelo ve los ejemplos,
# predice una clase, compara con la clase verdadera, calcula el error, y
# ajusta sus pesos internos para equivocarse un poco menos la próxima vez.
# Esto se repite miles de veces (una por cada lote, en cada época).
#
# QUÉ PASARÍA SI NO CORRIÉRAMOS ESTA CELDA: el modelo seguiría siendo
# el BETO original sin ajustar, que no sabe nada sobre distinguir texto
# humano de texto de Claude/ChatGPT/Gemini — respondería básicamente al azar.

entrenador.train()

In [ ]:
# ============================================================
# CELDA 11 — Evaluar el modelo ya entrenado
# ============================================================
# Corre el modelo sobre el grupo de prueba (que nunca vio durante el
# entrenamiento) y muestra accuracy y f1_macro reales.
#
# CÓMO LEER EL RESULTADO: un accuracy de 0.70, por ejemplo, significa que
# el modelo acertó la clase exacta (entre 4 posibles) en el 70% de los
# textos de prueba. Como hay 4 clases, "adivinar al azar" rondaría el 25%
# — así que cualquier número bien por encima de eso significa que el
# modelo aprendió señales reales, no que está tirando monedas.
#
# Con un dataset chico (decenas de ejemplos por clase, no miles), es
# esperable un accuracy modesto, no un 95%. Eso no es un error del
# notebook: es la limitación honesta de entrenar con pocos datos, y vale
# la pena decirlo así en la defensa del proyecto.

resultado_evaluacion = entrenador.evaluate()
print(resultado_evaluacion)

## Si el resultado te parece bajo

No te preocupes si el accuracy da algo como 0.40-0.60: con ~50 ejemplos
por clase es normal y esperable, no es un error. Para mejorar esto en el
futuro, la palanca más efectiva es **más datos**, no cambiar la
configuración de esta celda.

In [ ]:
# ============================================================
# CELDA 12 — Subir el modelo entrenado a Hugging Face
# ============================================================
# Hasta acá, el modelo entrenado solo existe en la memoria de esta sesión
# de Colab — si cerrás la pestaña, se pierde. push_to_hub lo sube como un
# repositorio de modelo en tu cuenta de Hugging Face, igual que cualquier
# otro modelo público, disponible para siempre y consultable por API.
#
# notebook_login() pide tu token de Hugging Face (el mismo que generaste
# en Settings > Access Tokens, con permiso de escritura esta vez, no solo
# lectura, porque vamos a CREAR un repositorio nuevo).
#
# QUÉ PASARÍA SI NO CORRIÉRAMOS ESTA CELDA: el modelo entrenado quedaría
# atrapado en esta sesión de Colab. detector.php no tendría forma de
# consultarlo, porque no existiría en ningún lado accesible por API.

from huggingface_hub import notebook_login
notebook_login()  # pegá acá tu token con permiso de escritura cuando lo pida

NOMBRE_DEL_MODELO = "vat-clasificador-es"  # vas a poder llamarlo tu-usuario/vat-clasificador-es

modelo.push_to_hub(NOMBRE_DEL_MODELO)
tokenizer.push_to_hub(NOMBRE_DEL_MODELO)

print(f"Listo. Tu modelo queda en: https://huggingface.co/TU_USUARIO/{NOMBRE_DEL_MODELO}")
print("Usá ese ID completo (usuario/nombre) en HF_CLASSIFIER_MODEL dentro de config.php")

## Próximo paso: conectar esto a `detector.php`

Una vez que termine de subirse, andá a `https://huggingface.co/TU_USUARIO/vat-clasificador-es`
para confirmar que el modelo quedó público. Después solo hay que cambiar
una línea en `config.php`:

```php
define('HF_CLASSIFIER_MODEL', 'TU_USUARIO/vat-clasificador-es');
```

Y ajustar el parseo en `detector.php`, porque este modelo ya devuelve
las 4 clases reales (`human`, `chatgpt`, `claude`, `gemini`) en vez del
reparto parejo que usábamos antes — avisame cuando llegues acá y lo
actualizamos juntos.